In [ ]:
# Лабораторная работа 5.1 
## Прогнозирование временных рядов с использованием RNN (LSTM) 

# ============================================================
# 1. ИМПОРТ БИБЛИОТЕК
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Проверка CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

In [ ]:
# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ ИЗ CSV ИЛИ ГЕНЕРАЦИЯ
# ============================================================

def load_data_from_csv(filepath, date_col=None, value_col=None, 
                       date_format='%Y-%m-%d'):
    """
    Функция для загрузки временного ряда из CSV-файла.
    
    Параметры:
    - filepath: путь к CSV-файлу
    - date_col: название колонки с датами (если None, используется индекс)
    - value_col: название колонки со значениями
    - date_format: формат даты (для парсинга)
        
    Возвращает:
    - pandas.Series с индексом из дат (если они есть)
    """
    # Подсказка: используйте df=pd.read_csv()
    # <-- ДОПОЛНИТЕ КОД
    pass
    
    return series    

def generate_synthetic_data(periods=730, trend=0.01, seasonal_amplitude=10, 
                           seasonal_period=30, noise_std=1.0):
    """
    Генерация синтетического временного ряда.
    """
    np.random.seed(42)
    t = np.linspace(0, periods, periods)
    trend_vals = trend * t
    seasonal = seasonal_amplitude * np.sin(2 * np.pi * t / seasonal_period)
    noise = np.random.normal(0, noise_std, size=t.shape)
    data = trend_vals + seasonal + noise
    return pd.Series(data, index=pd.date_range(start='2000-01-01', periods=periods, freq='D'))

#2.1. Загрузка данных из CSV (пример с температурой)
# Путь к файлу с данными (замените на свой путь)
# Пример данных: max_temperature.csv с колонками date,temperature

try:
    # Загрузить реальные данные
    series = load_data_from_csv(
        filepath='max_temperature.csv', 
        date_col='date',
        value_col='temperature',
        date_format='%Y-%m-%d'
    )
    print(f"Загружены реальные данные: {len(series)} записей")
    print(f"Период: {series.index[0]} — {series.index[-1]}")
    
except FileNotFoundError:
    print("Файл не найден. Используются синтетические данные.")
    series = generate_synthetic_data(periods=730)
    print(f"Сгенерированы синтетические данные: {len(series)} записей")

#2.2. Визуализация исходного ряда

plt.figure(figsize=(14, 5))
plt.plot(series.index, series.values, linewidth=1.5)
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.title('Исходный временной ряд')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()    


In [ ]:
# ============================================================
# 3. ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
# ============================================================

def create_sequences(data, lookback):
    """
    Преобразование временного ряда в набор последовательностей.
    
    Параметры:
    - data: массив значений (2D: (n_samples, 1))
    - lookback: количество предыдущих шагов
    
    Возвращает:
    - X: входные последовательности (n_samples - lookback, lookback, 1)
    - y: целевые значения (n_samples - lookback, 1)
    """
    X, y = [], []
    # ЗАДАНИЕ: Реализуйте создание последовательностей
    # for i in range(...):
    #     X.append(...)
    #     y.append(...)
    return np.array(X), np.array(y)

# Основной пайплайн обработки данных
def prepare_data(series, lookback=30, train_ratio=0.7, val_ratio=0.15):
    """
    Полная подготовка данных: нормализация, создание последовательностей, разделение.
    
    Возвращает:
    - X_train, y_train, X_val, y_val, X_test, y_test: тензоры PyTorch
    - scaler: объект MinMaxScaler (для обратного преобразования)
    - series_original: исходный ряд (для визуализации)
    """
    # Нормализация
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(series.values.reshape(-1, 1))
    
    # Создание последовательностей
    X, y = create_sequences(scaled_data, lookback)
    
    # Разделение
    total_len = len(X)
    train_size = int(total_len * train_ratio)
    val_size = int(total_len * val_ratio)
    
    X_train = X[:train_size]
    y_train = y[:train_size]
    X_val = X[train_size:train_size+val_size]
    y_val = y[train_size:train_size+val_size]
    X_test = X[train_size+val_size:]
    y_test = y[train_size+val_size:]
    
    # Преобразование в тензоры
    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.FloatTensor(y_train).to(device)
    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.FloatTensor(y_val).to(device)
    X_test_t = torch.FloatTensor(X_test).to(device)
    y_test_t = torch.FloatTensor(y_test).to(device)
    
    return (X_train_t, y_train_t, X_val_t, y_val_t, X_test_t, y_test_t), scaler, series

# Параметры
lookback = 30
train_ratio = 0.7
val_ratio = 0.15

# Подготовка данных
data_tensors, scaler, series_original = prepare_data(
    series, lookback=lookback, train_ratio=train_ratio, val_ratio=val_ratio
)

X_train_t, y_train_t, X_val_t, y_val_t, X_test_t, y_test_t = data_tensors

print(f"Обучающая выборка: {len(X_train_t)} примеров")
print(f"Валидационная выборка: {len(X_val_t)} примеров")
print(f"Тестовая выборка: {len(X_test_t)} примеров")



In [ ]:
# ============================================================
# 4. ОПРЕДЕЛЕНИЕ МОДЕЛИ LSTM
# ============================================================

class LSTMModel(nn.Module):
    """
    Модель LSTM для прогнозирования временных рядов.
    """
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # ЗАДАНИЕ: Определите слои LSTM и полносвязный слой
        # self.lstm = nn.LSTM(...)
        # self.fc = nn.Linear(...)
        
    def forward(self, x):
        # ЗАДАНИЕ: Реализуйте прямой проход
        # 1. Инициализация скрытого состояния
        # 2. Прямой проход через LSTM
        # 3. Взятие выхода последнего временного шага
        # 4. Полносвязный слой
        pass
        return out

# Создание модели
model = LSTMModel(input_size=1, hidden_size=64, num_layers=2, output_size=1).to(device)
print(model)

# Подсчёт параметров
total_params = sum(p.numel() for p in model.parameters())
print(f"Количество параметров: {total_params:,}")

In [ ]:
# ============================================================
# 5. ОБУЧЕНИЕ МОДЕЛИ
# ============================================================

# Функция для обучения модели
def train_lstm_model(model, X_train, y_train, X_val, y_val, 
                     num_epochs=100, batch_size=32, lr=0.001, verbose=True):
    """
    Обучение модели LSTM с валидацией.
    
    Возвращает:
    - model: обученная модель
    - train_losses, val_losses: история потерь
    """
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    train_losses = []
    val_losses = []
    
    
    # ЗАДАНИЕ: Реализуйте цикл обучения
    # for epoch in range(num_epochs):
    #     model.train()
    #     epoch_loss = 0
    #     for batch_X, batch_y in train_loader:
    #         ...
    #     model.eval()
    #     with torch.no_grad():
    #         val_outputs = model(X_val_t)
    #         val_loss = criterion(...)
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"Эпоха {epoch+1}/{num_epochs}, Train Loss: {train_losses[-1]:.6f}, Val Loss: {val_losses[-1]:.6f}")
    
    return model, train_losses, val_losses

#Обучение модели
model, train_losses, val_losses = train_lstm_model(
    model, X_train_t, y_train_t, X_val_t, y_val_t,
    num_epochs=100, batch_size=32, lr=0.001
)

# Визуализация потерь
plt.figure(figsize=(12, 4))
plt.plot(train_losses, label='Обучающая выборка')
plt.plot(val_losses, label='Валидационная выборка')
plt.xlabel('Эпоха')
plt.ylabel('Потери (MSE)')
plt.title('Динамика потерь при обучении')
plt.legend()
plt.grid(True)
plt.show()



In [ ]:
# ============================================================
# 6. ПРОГНОЗИРОВАНИЕ И ОЦЕНКА КАЧЕСТВА
# ============================================================

# Функция для прогнозирования и визуализации
def evaluate_and_plot(model, X_test, y_test, scaler, series, lookback=30):
    """
    Выполнение прогноза, расчёт метрик и визуализация.
    """
    model.eval()
    with torch.no_grad():
        y_pred = model(X_test).cpu().numpy()
        y_test_np = y_test.cpu().numpy()
    
    # Обратное масштабирование
    y_pred_orig = scaler.inverse_transform(y_pred)
    y_test_orig = scaler.inverse_transform(y_test_np)
    
    # Метрики
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
    mae = mean_absolute_error(y_test_orig, y_pred_orig)
    
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    
    # Относительные метрики
    y_mean = np.mean(y_test_orig)
    print(f"Относительная RMSE: {rmse / y_mean * 100:.2f}%")
    print(f"Относительная MAE: {mae / y_mean * 100:.2f}%")
       
    # Восстановление индексов для тестовой части
    test_start_idx = len(series) - len(y_test_orig)
    test_indices = series.index[test_start_idx:test_start_idx + len(y_test_orig)]
    
    # Сдвиг для визуализации (без сдвига для расчёта метрик)
    y_pred_shifted = y_pred_orig[1:]  # Убираем первый предсказанный день
    y_test_shifted = y_test_orig[:-1]   # Убираем последний реальный день
    
    # Визуализация со сдвигом для анализа формы
    plt.figure(figsize=(14, 6))
    plt.plot(y_test_shifted, label='Реальные', linewidth=2)
    plt.plot(y_pred_shifted, label='Предсказанные (со сдвигом)', linewidth=2, linestyle='--')
    plt.xlabel('Шаг (дни)')
    plt.ylabel('Температура (°C)')
    plt.title('Прогноз со сдвигом (видно совпадение тренда и сезонности)')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show() 
  
    return rmse, mae

# Оценка и визуализация прогноза
rmse, mae = evaluate_and_plot(model, X_test_t, y_test_t, scaler, series_original, lookback=lookback)

In [ ]:
# 7.Исследование влияния lookback
# lookbacks = [10, 20, 30, 60]
# ... (реализуйте самостоятельно)


In [ ]:
# ============================================================
# 8. ДОПОЛНИТЕЛЬНОЕ ЗАДАНИЕ
# ============================================================

#  Сравнение LSTM и GRU
# ... (реализуйте самостоятельно)

In [ ]:
# ============================================================
# 8. ВЫВОДЫ
# ============================================================

print("\n=== ИТОГОВЫЕ РЕЗУЛЬТАТЫ ===")
print(f"Источник данных: {filepath}")
print(f"Размер данных: {len(series)} записей")
print(f"lookback: {lookback}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Относительная RMSE: {rmse / np.mean(y_test_orig) * 100:.2f}%")

# Сохранение модели (опционально)
# torch.save(model.state_dict(), 'lstm_model.pth')